### 1. Import Libraries

In [1]:
import pandas as pd

### 2. Load and Consolidate Data

Load the leptospirosis.xlsx file. Iterate through each sheet (representing a year) and extract the district wise monthly case data. Consolidate all this information into a single pandas Dataframe with columns for 'Year', 'Month', 'District', and 'Cases'.

In [2]:
# Define the path to the Excel file
file_path = '../../data/source/socioeconomic.xlsx'

# Load the Excel file
socioeconomic_source_df = pd.read_excel(file_path)
socioeconomic_source_df

,District,Population_2012,Households_2012,Population_2024,Households_2024
0,Colombo,2324349,566524,2375415,654051
1,Gampaha,2304833,598420,2436142,683025
2,Kalutara,1221948,301544,1305784,349430
3,Kandy,1375382,342115,1461895,389826
4,Matale,484531,127578,526870,148184
5,Nuwara Eliya,711644,176113,725280,192915
6,Galle,1063334,269362,1097372,305379
7,Matara,814048,203763,837889,230034
8,Hambantota,599903,155988,671418,187558
9,Jaffna,583882,136074,594751,156797


### 3. Processing

 - Interpolate data using Compound Annual Growth Rate (CAGR) Interpolation

##### 3.1. CAGR Calculations (Rate)

In [3]:
# Function to calculate CAGR
def calculate_cagr(start_value, end_value, years):
    """Calculate Compound Annual Growth Rate"""
    if start_value == 0 or end_value == 0:
        return 0
    return (end_value / start_value) ** (1 / years) - 1


# Function to interpolate values using CAGR
def interpolate_value(base_value, cagr, years_from_base):
    """Interpolate value using CAGR"""
    return base_value * (1 + cagr) ** years_from_base


# Calculate CAGR for 2012-2024 (12 years)
socioeconomic_source_df['Pop_CAGR_2012_2024'] = socioeconomic_source_df.apply(
    lambda row: calculate_cagr(row['Population_2012'], row['Population_2024'], 12), axis=1
)
socioeconomic_source_df['HH_CAGR_2012_2024'] = socioeconomic_source_df.apply(
    lambda row: calculate_cagr(row['Households_2012'], row['Households_2024'], 12), axis=1
)


##### 3.2. Year wise population and households
 - For 2007-2011, checked backwards from 2012 using the same growth rate
 - This assumes similar growth patterns before and after 2012

In [4]:
# Create a dataset with all years from 2007 to 2024
years = list(range(2007, 2025))
interpolated_data = []

for _, row in socioeconomic_source_df.iterrows():
    district = row['District']
    
    for year in years:
        years_from_2012 = year - 2012
        
        # Use the 2012-2024 CAGR for all interpolations
        pop_estimate = interpolate_value(
            row['Population_2012'], 
            row['Pop_CAGR_2012_2024'], 
            years_from_2012
        )
        
        hh_estimate = interpolate_value(
            row['Households_2012'], 
            row['HH_CAGR_2012_2024'], 
            years_from_2012
        )
        
        # For actual survey years, use exact values
        if year == 2012:
            pop_estimate = row['Population_2012']
            hh_estimate = row['Households_2012']
        elif year == 2024:
            pop_estimate = row['Population_2024']
            hh_estimate = row['Households_2024']
        
        interpolated_data.append({
            'District': district,
            'Year': year,
            'Population': int(round(pop_estimate)),
            'Households': int(round(hh_estimate)),
            'Avg_HH_Size': round(pop_estimate / hh_estimate, 2) if hh_estimate > 0 else 0
        })


# Create final dataframe
processed_socioeconomic_df = pd.DataFrame(interpolated_data)

In [5]:
processed_socioeconomic_df

,District,Year,Population,Households,Avg_HH_Size
0,Colombo,2007,2303397,533606,4.32
1,Colombo,2008,2307572,540033,4.27
2,Colombo,2009,2311755,546538,4.23
3,Colombo,2010,2315945,553120,4.19
4,Colombo,2011,2320143,559782,4.14
...,...,...,...,...,...
445,Kegalle,2020,860418,234016,3.68
446,Kegalle,2021,862921,235993,3.66
447,Kegalle,2022,865432,237988,3.64
448,Kegalle,2023,867950,239999,3.62


In [6]:
district_mapping = {

    "Ampara": "Ampara",
    "Anuradhapura": "Anuradhapura",
    "Badulla": "Badulla",
    "Batticaloa": "Batticaloa",
    "Colombo": "Colombo",
    "Galle": "Galle",
    "Gampaha": "Gampaha",
    "Hambantota": "Hambantota",
    "Jaffna": "Jaffna",
    "Kalutara": "Kalutara",
    "Kandy": "Kandy",
    "Kegalle": "Kegalle",
    "Kilinochchi": "Kilinochchi",
    "Kurunegala": "Kurunegala",
    "Mannar": "Mannar",
    "Matale": "Matale",
    "Matara": "Matara",
    "Moneragala": "Monaragala",
    "Mullaitivu": "Mullaitivu",
    "Nuwara Eliya": "Nuwara Eliya",
    "Polonnaruwa": "Polonnaruwa",
    "Puttalam": "Puttalam",
    "Ratnapura": "Ratnapura",
    "Trincomalee": "Trincomalee",
    "Vavuniya": "Vavuniya"
}

In [7]:
processed_socioeconomic_df["District"] = processed_socioeconomic_df["District"].map(district_mapping)


In [8]:
processed_socioeconomic_df

,District,Year,Population,Households,Avg_HH_Size
0,Colombo,2007,2303397,533606,4.32
1,Colombo,2008,2307572,540033,4.27
2,Colombo,2009,2311755,546538,4.23
3,Colombo,2010,2315945,553120,4.19
4,Colombo,2011,2320143,559782,4.14
...,...,...,...,...,...
445,Kegalle,2020,860418,234016,3.68
446,Kegalle,2021,862921,235993,3.66
447,Kegalle,2022,865432,237988,3.64
448,Kegalle,2023,867950,239999,3.62


#### 3.3. Data Quality Check

In [9]:
district_count_per_year = (
    processed_socioeconomic_df
    .groupby(["Year"])["District"]
    .nunique()
    .reset_index(name="District_Count")
)


year_count_per_district = (
    processed_socioeconomic_df
    .groupby("District")[["Year"]]
    .apply(lambda x: x.drop_duplicates().shape[0])
    .reset_index(name="Month_Count")
)




In [11]:
##Check whether all the data are available
district_count_per_year.to_csv('../../data/processed/data_quality_check/socioeconomic_district_count_per_year.csv',
                index=False)

##Check whether all the data are available
year_count_per_district.to_csv('../../data/processed/data_quality_check/socioeconomic_year_count_per_district.csv',
                index=False)

### Save Processed Data
 - data\processed

In [12]:
##saved as a new csv
processed_socioeconomic_df.to_csv('../../data/processed/4_socioeconomic_processed.csv',
                index=False)